# 🧠 LLM Starter Course
### A hands-on introduction for developers

**Model:** Gemma 4 31B (via Google Generative AI API)  
**What you'll learn:**
1. What an LLM actually is (no math, developer analogies)
2. Tokens & tokenization
3. Your first API call — and how to read the response
4. Core sampling parameters: `temperature`, `top_k`, `top_p`, `max_output_tokens`, `stop_sequences`
5. System instructions
6. Multi-turn chat & conversation context
7. Prompt engineering patterns: zero-shot, few-shot, chain-of-thought
8. Structured output (JSON)
9. Streaming responses
10. Embeddings & semantic similarity
11. Hallucination & grounding
12. Context window limits
13. Multiple candidates
14. Multimodal input (vision)
15. Prompt templates & parameterization

---
> **Prerequisites:** Python basics. No ML experience needed.
>
> **Note on Gemma 4:** This is a *thinking model* — it reasons internally before answering. Thinking tokens count against `max_output_tokens`, so we use generous limits (2048+) throughout.

---
## 0. Setup

In [ ]:
%pip install -q google-genai python-dotenv

In [ ]:
import os
import json
import math
import time
import base64
import getpass
import urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API key from .env file (falls back to env var, then interactive prompt)
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL = "gemma-4-31b-it"

# Gemma 4 is a thinking model: it reasons internally before answering.
# Thinking tokens count against max_output_tokens, so we use 2048 as the
# default throughout (thinking typically consumes 100-1400 tokens).
DEFAULT_MAX = 2048

def cfg(**kwargs):
    """Shorthand for GenerateContentConfig with safe defaults."""
    kwargs.setdefault("max_output_tokens", DEFAULT_MAX)
    kwargs.setdefault("temperature", 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    """Safely extract text from a response (handles None from thinking models)."""
    if response.text:
        return response.text.strip()
    # Fallback: collect non-thought parts manually
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return "".join(parts).strip()

print("✅ Setup complete. Model:", MODEL)

### 🔌 API Key Test

In [ ]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

---
## 1. What Is an LLM?

An LLM (Large Language Model) is, at heart, a **next-token predictor** trained on massive amounts of text.

At every step it asks: *"Given everything so far, what token most likely comes next?"*

```
Input text
    │
    ▼
┌─────────────┐
│  Tokenizer  │  "The cat sat" → [450, 4097, 2492]
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Transformer │  Billions of parameters, trained on the internet
│   (the LLM) │
└──────┬──────┘
       │
       ▼
┌────────────────────────────────┐
│ Probability distribution over │  " on" → 42%
│ the entire vocabulary (~32k    │  " in" → 18%
│ possible next tokens)          │  " near" → 9%  ...
└──────────────┬─────────────────┘
               │
               ▼
         Sample a token → append → repeat until done
```

**Developer analogy:** Think of it as autocomplete on steroids — trained on the whole internet. It doesn't "know" facts the way a database does; it learned statistical patterns so well that it *acts as if* it knows them.

| Concept | What it means in practice |
|---|---|
| Probabilistic output | Same prompt → different answer each time (unless `temperature=0`) |
| Context window | Model can only "see" a fixed number of tokens at once |
| No memory by default | Each API call is stateless; you send the whole conversation each time |
| Thinking models | Some models (like Gemma 4) reason internally before answering — improves quality |

---
## 2. Tokens & Tokenization

Tokens are **not** words, and not characters — they're chunks roughly 3–4 characters long on average.

**Why it matters:** you pay per token; context windows are measured in tokens; speed depends on token count.

In [ ]:
sentences = [
    "Hello",
    "Hello, world!",
    "Hello, world! How are you doing today?",
    "Supercalifragilisticexpialidocious",
    "The quick brown fox jumps over the lazy dog.",
    "Python is great for data science, machine learning, and automation.",
    "🚀🔥💡",
    "\n\n\n\n",
]

print(f"{'Text':<60} {'Tokens':>6}")
print("-" * 68)
for s in sentences:
    result = client.models.count_tokens(model=MODEL, contents=s)
    display_s = repr(s) if len(s) < 55 else repr(s[:52]) + "..."
    print(f"{display_s:<60} {result.total_tokens:>6}")

In [ ]:
# ✏️  Change this string and re-run to experiment
my_text = "Write your own sentence here and see how many tokens it uses!"

result = client.models.count_tokens(model=MODEL, contents=my_text)
tokens = result.total_tokens
print(f"Text   : {my_text}")
print(f"Words  : {len(my_text.split())}")
print(f"Chars  : {len(my_text)}")
print(f"Tokens : {tokens}")
print(f"Ratio  : {len(my_text)/tokens:.1f} chars/token  |  {len(my_text.split())/tokens:.1f} words/token")

---
## 3. Your First API Call

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is a neural network? Explain in 2 sentences for a software developer.",
    config=cfg()
)
print(get_text(response))

In [ ]:
# Dissect the response object
print("=" * 60)
print("RESPONSE OBJECT ANATOMY")
print("=" * 60)

text = get_text(response)
print("\n--- text ---")
print(text[:200], "..." if len(text) > 200 else "")

print("\n--- finish_reason ---")
print(f"  {response.candidates[0].finish_reason}")

print("\n--- usage_metadata ---")
u = response.usage_metadata
print(f"  prompt_token_count    : {u.prompt_token_count}")
print(f"  candidates_token_count: {u.candidates_token_count}")
print(f"  thoughts_token_count  : {getattr(u, 'thoughts_token_count', 'N/A')}")
print(f"  total_token_count     : {u.total_token_count}")

---
## 4. Core Sampling Parameters

```
All vocabulary tokens (~32k)
         │
    top_k filter  ──→  keep only top K tokens by probability
         │
    top_p filter  ──→  keep smallest set whose cumulative prob ≥ p
         │
  temperature     ──→  sharpen (< 1) or flatten (> 1) the distribution
         │
      SAMPLE  ──→  pick one token  ──→  append  ──→  repeat
```
The filters stack: `top_k` first, then `top_p`, then `temperature` reshapes what's left.

### 4a. `temperature` — creativity dial

| Value | Behaviour |
|---|---|
| `0.0` | Always pick the highest-probability token (deterministic) |
| `0.7` | Balanced — good default |
| `1.0` | Raw model probabilities, no reshaping |
| `2.0` | Distribution nearly flat — very random |

**Analogy:** Job candidates ranked by score. `temperature=0` → always hire rank #1. `temperature=2` → shuffle the entire list randomly.

In [ ]:
prompt = "Continue this story in ONE sentence only: 'The robot opened the door and saw'"

for temp in [0.0, 0.7, 1.5]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=temp)
    )
    print(f"temperature={temp}: {get_text(r)}")
    print()

In [ ]:
# temperature=0 → near-identical results across runs
print("Same prompt × 3 at temperature=0:")
for i in range(3):
    r = client.models.generate_content(
        model=MODEL, contents="What is 17 × 8? Reply with only the number.",
        config=cfg(temperature=0.0)
    )
    print(f"  Run {i+1}: {get_text(r)}")

### 4b. `top_k` — vocabulary hard limit

Before sampling, **discard all but the top-k tokens** by probability.

| Value | Effect |
|---|---|
| `1` | Greedy — always pick the most probable token |
| `10` | Only consider the 10 most likely tokens |
| `40` | Typical default |

**Analogy:** Only put the top-k most-likely words in a hat before drawing.

In [ ]:
prompt = "Reply with ONLY a single color name, nothing else:"
for k in [1, 5, 100]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0, top_k=k)
    )
    print(f"top_k={k:>3}: {get_text(r)}")

In [ ]:
# top_k=1 → same token every time (greedy)
print("top_k=1 repeated 3 times (expect same answer):")
for _ in range(3):
    r = client.models.generate_content(
        model=MODEL,
        contents="The most widely-used programming language is (one word only):",
        config=cfg(top_k=1, temperature=0.0)
    )
    print(" ", get_text(r))

### 4c. `top_p` — nucleus sampling

Keep the **smallest set of tokens whose cumulative probability ≥ p**.

```
"the"   → 55%  cumulative=55%  keep
"a"     → 20%  cumulative=75%  keep
"this"  → 10%  cumulative=85%  keep
"that"  →  8%  cumulative=93%  ← STOP (crossed 0.9)
```

| Value | Effect |
|---|---|
| `0.1` | Ultra-narrow: only top tokens |
| `0.9` | Common default |
| `1.0` | No filtering |

In [ ]:
prompt = "Give ONE synonym for 'happy' (single word only):"
for p in [0.05, 0.5, 1.0]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0, top_p=p)
    )
    print(f"top_p={p}: {get_text(r)}")

### 4d. `max_output_tokens` — length hard cap

Stops generation after this many tokens. It's a **budget**, not a target.

> ⚠️ **Gemma 4 note:** Because Gemma 4 uses thinking tokens internally, you need a higher `max_output_tokens` than you might expect. `DEFAULT_MAX=2048` is used throughout this notebook.

In [ ]:
prompt = "Explain what machine learning is."
for limit in [300, 700, 2048]:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=0.5, max_output_tokens=limit)
    )
    finish = r.candidates[0].finish_reason
    used = r.usage_metadata.candidates_token_count or 0
    thoughts = getattr(r.usage_metadata, 'thoughts_token_count', 0) or 0
    text = get_text(r)
    print(f"--- max={limit} (thinking={thoughts}, answer={used}, finish={finish.name}) ---")
    print(text[:200], "..." if len(text) > 200 else "")
    print()

| `finish_reason` | Meaning |
|---|---|
| `STOP` | Model finished naturally |
| `MAX_TOKENS` | Cut off — increase `max_output_tokens` |
| `SAFETY` | Blocked by safety filters |

### 4e. `stop_sequences` — custom stop strings

Stop generation when one of these strings is produced. Great for structured extraction.

In [ ]:
prompt = "List three programming languages, one per line, numbered 1. 2. 3.:"

r_full = client.models.generate_content(
    model=MODEL, contents=prompt, config=cfg(temperature=0.0)
)
print("Without stop_sequences:")
print(get_text(r_full))
print()

r_stopped = client.models.generate_content(
    model=MODEL, contents=prompt,
    config=cfg(temperature=0.0, stop_sequences=["3."])
)
print("With stop_sequences=['3.'] — stops before item 3:")
print(repr(get_text(r_stopped)))

In [ ]:
# Practical use: extract just the country name, stop before any explanation
r = client.models.generate_content(
    model=MODEL,
    contents="What country is Paris in? Answer with ONLY the country name, nothing else.",
    config=cfg(temperature=0.0, stop_sequences=[".", "\n"])
)
print("Extracted country:", get_text(r))

### 4f. Parameter Quick-Reference

| Task | temperature | top_k | top_p |
|---|---|---|---|
| Factual Q&A / extraction | 0.0–0.2 | 1–10 | 0.1–0.5 |
| Balanced chat | 0.7 | 40 | 0.9 |
| Creative writing | 1.0–1.5 | 40–100 | 0.95–1.0 |
| Brainstorming | 1.5–2.0 | 100+ | 1.0 |

---
## 5. System Instructions

A privileged prompt that sets the model's **persona, rules, and constraints** before the conversation starts. Users never see it.

In [ ]:
user_question = "What is 10 divided by 2?"

personas = {
    "No system instruction": None,
    "Pirate": "You are a pirate. Answer in pirate dialect. Start with 'Arrr!'.",
    "Professor": "You are a formal mathematics professor. Use precise academic language.",
    "5-year-old teacher": "You explain everything to a 5-year-old. Use very simple words and a fun analogy.",
}

for name, sys_inst in personas.items():
    r = client.models.generate_content(
        model=MODEL, contents=user_question,
        config=cfg(temperature=0.7, system_instruction=sys_inst)
    )
    print(f"[{name}]")
    print(get_text(r)[:200])
    print()

In [ ]:
# System instructions can enforce strict output constraints
r = client.models.generate_content(
    model=MODEL,
    contents="Tell me about Python.",
    config=cfg(
        temperature=0.5,
        system_instruction=(
            "You are a concise assistant. "
            "RULES: 1) Always answer in exactly 3 bullet points. "
            "2) Each bullet must be under 15 words. "
            "3) Never use the word 'powerful'."
        )
    )
)
print(get_text(r))

---
## 6. Multi-Turn Chat & Conversation Context

LLMs are **stateless** — each API call is independent. To have a conversation you must send the **full history** every time. The SDK's `chat` object manages this automatically.

```
Turn 1:  [user msg 1]  →  model
Turn 2:  [user msg 1 + model reply 1 + user msg 2]  →  model
Turn 3:  [full history + user msg 3]  →  model
```

In [ ]:
chat = client.chats.create(
    model=MODEL,
    config=cfg(
        system_instruction="You are a helpful coding tutor. Be encouraging and concise.",
        temperature=0.7
    )
)

r1 = chat.send_message("Hi! I'm learning Python and heard the word 'list'. What is it?")
print("User: Hi! I'm learning Python and heard the word 'list'. What is it?")
print(f"Model: {get_text(r1)}\n")

# Turn 2 — chat object automatically includes history
r2 = chat.send_message("Can you show me how to add an item to it?")
print("User: Can you show me how to add an item to it?")
print(f"Model: {get_text(r2)}")

In [ ]:
# Watch how token usage GROWS with each turn — this is why context windows matter
chat2 = client.chats.create(model=MODEL, config=cfg(temperature=0.5))

messages = [
    "My name is Bob.",
    "I live in Tokyo.",
    "I work as a software engineer.",
    "What do you know about me so far? Be brief.",
]

print(f"{'Turn':<6} {'Prompt tokens':>15} {'Output tokens':>14} {'Thinking':>10} {'Total':>8}")
print("-" * 58)
for i, msg in enumerate(messages, 1):
    r = chat2.send_message(msg)
    u = r.usage_metadata
    t = getattr(u, 'thoughts_token_count', 0) or 0
    print(f"{i:<6} {u.prompt_token_count:>15} {(u.candidates_token_count or 0):>14} {t:>10} {u.total_token_count:>8}")

print("\nFinal response:", get_text(r))

---
## 7. Prompt Engineering Patterns

| Pattern | What it is | When to use |
|---|---|---|
| **Zero-shot** | Just ask, no examples | Simple well-known tasks |
| **Few-shot** | Include 2–5 examples | Specific format or rare classification |
| **Chain-of-thought** | Ask to reason step-by-step | Math, logic, multi-step problems |

In [ ]:
# Zero-shot — just ask
r = client.models.generate_content(
    model=MODEL,
    contents="Classify as POSITIVE, NEGATIVE, or NEUTRAL (one word only):\nReview: 'The battery life is mediocre but the camera is stunning.'",
    config=cfg(temperature=0.0)
)
print("Zero-shot:", get_text(r))

In [ ]:
# Few-shot — examples teach the exact output format
few_shot_prompt = """Classify sentiment. Reply with ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL.

Review: "This laptop is blazing fast and the build quality is excellent."
Sentiment: POSITIVE

Review: "Terrible customer service. Waited 3 weeks and got the wrong item."
Sentiment: NEGATIVE

Review: "It arrived on time. Does what it says on the box."
Sentiment: NEUTRAL

Review: "The battery life is mediocre but the camera is stunning."
Sentiment:"""

r_few = client.models.generate_content(
    model=MODEL, contents=few_shot_prompt,
    config=cfg(temperature=0.0, stop_sequences=["\n"])
)
print("Few-shot:", get_text(r_few))

In [ ]:
# Chain-of-thought — compare direct vs step-by-step
hard_question = (
    "A store has 3 shelves. Each shelf has 4 rows. Each row has 8 items. "
    "On Monday, 20% of items are sold. On Tuesday, 15 more items are added. "
    "How many items are in the store at the end of Tuesday?"
)

r_direct = client.models.generate_content(
    model=MODEL,
    contents=hard_question + "\nAnswer with ONLY a number, nothing else.",
    config=cfg(temperature=0.0)
)
print("Direct (no CoT):", get_text(r_direct))

r_cot = client.models.generate_content(
    model=MODEL,
    contents=hard_question + "\nThink step by step, then give the final answer.",
    config=cfg(temperature=0.0, max_output_tokens=4096)
)
print("\nChain-of-thought:")
print(get_text(r_cot))

# Manual verification
total = 3 * 4 * 8
after_monday = int(total * 0.80)
after_tuesday = after_monday + 15
print(f"\n✅ Manual check: {total} → {after_monday} → {after_tuesday}")

---
## 8. Structured Output (JSON)

Force the model to output valid JSON matching a schema — essential for production pipelines.

In [ ]:
invoice_text = """
Invoice from Acme Corp dated March 15 2024.
Customer: TechStartup Inc, 500 Main St, San Francisco CA.
Items: 10 units of Widget Pro at $49.99 each, plus shipping $12.50.
Total due: $512.40. Payment due within 30 days.
"""

schema = {
    "type": "object",
    "properties": {
        "vendor": {"type": "string"},
        "customer": {"type": "string"},
        "date": {"type": "string"},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "quantity": {"type": "number"},
                    "unit_price": {"type": "number"}
                }
            }
        },
        "shipping": {"type": "number"},
        "total": {"type": "number"},
        "payment_due_days": {"type": "number"}
    }
}

r = client.models.generate_content(
    model=MODEL,
    contents=f"Extract invoice details as JSON:\n{invoice_text}",
    config=cfg(temperature=0.0, response_mime_type="application/json", response_schema=schema)
)

data = json.loads(get_text(r))
print(json.dumps(data, indent=2))

In [ ]:
# The result is real Python data — access fields directly
print(f"Vendor  : {data['vendor']}")
print(f"Customer: {data['customer']}")
print(f"Total   : ${data['total']}")
for item in data.get('items', []):
    print(f"  - {item.get('description','?')}: {item.get('quantity','?')} × ${item.get('unit_price','?')}")

---
## 9. Streaming Responses

By default the API waits until the **full response is ready** before returning. Streaming sends tokens back as they're generated — like ChatGPT's typing effect.

```
Non-streaming:  [........generating........] → receive everything at once
Streaming:      [token] [token] [token] ...  → receive as produced
```

Use streaming whenever the response will be longer than a sentence, or users are waiting for output.

In [ ]:
# Non-streaming — we wait until everything is done
start = time.time()
r = client.models.generate_content(
    model=MODEL,
    contents="Write a 4-line poem about Python programming.",
    config=cfg(temperature=0.8)
)
elapsed = time.time() - start
print(f"Non-streaming: received after {elapsed:.1f}s")
print(get_text(r))

In [ ]:
# Streaming — tokens arrive in real-time, print as they come
print("Streaming (tokens arrive in real-time):")
print("-" * 40)

chunk_count = 0
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Write a 4-line poem about Python programming.",
    config=cfg(temperature=0.8)
):
    # Gemma 4 streams thinking chunks (thought=True) before answer chunks
    # We skip thought chunks so only the final answer prints
    if chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            if not getattr(part, 'thought', False) and part.text:
                print(part.text, end="", flush=True)
                chunk_count += 1

print(f"\n\nReceived {chunk_count} answer chunks")

In [ ]:
# Collecting streamed output into a single string (common pattern in apps)
full_text = ""
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="List 3 Python best practices, numbered.",
    config=cfg(temperature=0.3)
):
    if chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            if not getattr(part, 'thought', False) and part.text:
                full_text += part.text
                print(part.text, end="", flush=True)

print(f"\n\nTotal characters collected: {len(full_text)}")

---
## 10. Embeddings & Semantic Similarity

LLMs can also **represent text as numbers** — this is called an **embedding**.

An embedding is a list of ~3072 numbers (a vector) that captures the *meaning* of a piece of text.

```
"king"   → [0.23, -0.71, 0.45, ...  3072 numbers]
"queen"  → [0.21, -0.69, 0.43, ...  3072 numbers]  ← very similar!
"banana" → [-0.54, 0.12, -0.88, ... 3072 numbers]  ← very different
```

**Cosine similarity:** 1.0 = identical meaning, 0.0 = unrelated, −1.0 = opposite.

**Use cases:** semantic search, clustering, recommendation, RAG (retrieval-augmented generation).

In [ ]:
EMBED_MODEL = "gemini-embedding-001"

def embed(text: str) -> list:
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

def cosine_similarity(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b)

sentences = [
    "I love programming in Python.",
    "Python is my favourite coding language.",
    "The weather today is sunny and warm.",
    "Machine learning uses data to train models.",
    "Deep learning is a subset of machine learning.",
]

embeddings = {s: embed(s) for s in sentences}
print(f"Embedding dimension: {len(list(embeddings.values())[0])}")
print(f"First 5 values: {list(embeddings.values())[0][:5]}")

In [ ]:
# Similarity matrix — high values = similar meaning
labels = [s[:38] + "..." if len(s) > 38 else s for s in sentences]
max_len = max(len(l) for l in labels)

print("Cosine similarity (1.0 = identical meaning):\n")
header = " " * (max_len + 2) + "  ".join(f"[{i+1}]" for i in range(len(sentences)))
print(header)
for i, s1 in enumerate(sentences):
    row = f"[{i+1}] {labels[i]:<{max_len}}  "
    for s2 in sentences:
        sim = cosine_similarity(embeddings[s1], embeddings[s2])
        row += f"{sim:.2f}  "
    print(row)

In [ ]:
# Semantic search: find the most similar sentence to a query
def semantic_search(query: str, corpus: list) -> list:
    q_vec = embed(query)
    scores = [(cosine_similarity(q_vec, embed(doc)), doc) for doc in corpus]
    return sorted(scores, reverse=True)

query = "What coding language should I learn?"
results = semantic_search(query, sentences)

print(f"Query: '{query}'\n")
for score, doc in results:
    print(f"  {score:.3f}  {doc}")

---
## 11. Hallucination & Grounding

### What is hallucination?

LLMs **confidently generate false information**. This isn't a bug — it's a consequence of how they work. The model predicts the most *plausible-sounding* tokens, not the most *factually correct* ones.

Common patterns: invented citations, wrong numbers, fabricated events, confident but incorrect technical details.

### Mitigation strategies
| Strategy | How |
|---|---|
| **Lower temperature** | More deterministic = less likely to fabricate |
| **Grounding** | Provide the facts in the prompt; instruct model to use only those |
| **Verification step** | Second prompt: "Is this verifiable? Rate your confidence" |
| **RAG** | Retrieve real documents, include them in the prompt |

In [ ]:
# Ask about something that doesn't exist — the model may hallucinate details
r = client.models.generate_content(
    model=MODEL,
    contents=(
        "Who won the fictional 'Golden Syntax Award' for best Python library in 2023? "
        "Give details about the award ceremony."
    ),
    config=cfg(temperature=0.7)
)
print("Hallucination-prone prompt (award doesn't exist):")
print(get_text(r))

In [ ]:
# Grounding: provide the facts, instruct the model to use only them
context = """
COMPANY POLICY DOCUMENT (Q1 2024):
- Remote work allowance: up to 3 days per week
- Annual leave: 25 days per year
- Health insurance: covers employee and up to 2 dependants
- Equipment budget: $1,500 per year
"""

grounded_prompt = f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

DOCUMENT:
{context}

QUESTION: How many days of remote work are allowed per week?"""

r_grounded = client.models.generate_content(
    model=MODEL, contents=grounded_prompt,
    config=cfg(temperature=0.0)
)
print("Grounded answer:")
print(get_text(r_grounded))

In [ ]:
# Test: question NOT in the document — should refuse to answer
grounded_prompt2 = f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

DOCUMENT:
{context}

QUESTION: What is the company's parental leave policy?"""

r2 = client.models.generate_content(
    model=MODEL, contents=grounded_prompt2,
    config=cfg(temperature=0.0)
)
print("Out-of-scope question:")
print(get_text(r2))

In [ ]:
# Verification step: ask the model to rate its own confidence
claim = "The Python programming language was created in 1991 by Guido van Rossum."

r_check = client.models.generate_content(
    model=MODEL,
    contents=f"""Fact-check this claim. Rate confidence as HIGH, MEDIUM, or LOW. Explain any uncertainties.

Claim: {claim}""",
    config=cfg(temperature=0.0)
)
print(get_text(r_check))

---
## 12. Context Window Limits

Every model has a **context window** — the maximum tokens it can process (prompt + response combined).

| Model | Context window |
|---|---|
| Gemma 4 31B | ~128k tokens |
| Gemini 2.5 Flash | 1M tokens |

~128k tokens ≈ ~100,000 words ≈ a full novel.

### Strategies for long documents
| Strategy | When to use |
|---|---|
| **Chunking** | Split document, process each chunk separately |
| **Summarization chain** | Summarize each chunk, then summarize the summaries |
| **RAG** | Embed chunks, retrieve only the relevant ones |

In [ ]:
# Measure token counts for different document sizes
sentence = "The quick brown fox jumps over the lazy dog. "

print(f"{'~Words':>10} {'Tokens':>10} {'% of 128k window':>20}")
print("-" * 45)
for n_words in [100, 1_000, 10_000, 50_000]:
    text = sentence * (n_words // 9)
    result = client.models.count_tokens(model=MODEL, contents=text)
    tokens = result.total_tokens
    pct = tokens / 128_000 * 100
    print(f"{n_words:>10,} {tokens:>10,} {pct:>19.1f}%")

In [ ]:
# Chunking strategy: split a long text, summarize each chunk, combine
long_text = """
Chapter 1: Python was created by Guido van Rossum in 1991. It emphasizes readability 
and simplicity. The name comes from Monty Python, not the snake.

Chapter 2: Python has a rich ecosystem of libraries. NumPy handles numerical computation,
Pandas is for data manipulation, and Flask/Django are popular web frameworks.

Chapter 3: Python is widely used in data science, machine learning, web development,
automation, and scientific computing. It consistently ranks as one of the most 
popular programming languages worldwide.
"""

chunks = [c.strip() for c in long_text.strip().split("\n\n") if c.strip()]

# Step 1: summarize each chunk
chunk_summaries = []
for i, chunk in enumerate(chunks, 1):
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Summarize in ONE sentence:\n{chunk}",
        config=cfg(temperature=0.0)
    )
    summary = get_text(r)
    chunk_summaries.append(summary)
    print(f"Chunk {i}: {summary}")

# Step 2: combine summaries
r_final = client.models.generate_content(
    model=MODEL,
    contents="Combine these summaries into one paragraph:\n" + "\n".join(chunk_summaries),
    config=cfg(temperature=0.3)
)
print("\nFinal combined summary:")
print(get_text(r_final))

---
## 13. Generating Multiple Responses (Best-of-N)

Sometimes you want **N independent responses** to the same prompt — then pick the best one.
This is called **best-of-N sampling** and is a simple but powerful quality-improvement technique.

Why it helps:
- **Diversity** — higher temperature gives varied outputs; choose the one that fits best
- **Reliability** — if most candidates agree, you have more confidence in the answer
- **Evaluation** — compare different phrasings or approaches side by side

> **Note:** Some API providers support `candidate_count` to get multiple responses in one call.
> With Gemma 4 we simply call the API N times — same effect, slightly more tokens.

In [ ]:
# Generate 3 different taglines by calling the API 3 times at high temperature
prompt = "Write ONLY a single marketing tagline (no preamble) for a Python learning app called 'PyLeap'."
N = 3

candidates = []
for i in range(N):
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.2)  # high temperature for variety
    )
    candidates.append(get_text(r))

print(f"Generated {N} candidates:\n")
for i, text in enumerate(candidates, 1):
    print(f"Candidate {i}: {text}")

In [ ]:
# Best-of-N: generate N subject lines, then use the model to judge the best
prompt = "Write ONLY a single email subject line (no preamble) announcing a new Python course."

candidates_text = []
for _ in range(3):
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0)
    )
    candidates_text.append(get_text(r))

print("Candidates:")
for i, t in enumerate(candidates_text, 1):
    print(f"  {i}. {t}")

# Ask the model to pick the best one
selection_prompt = "Which subject line is most compelling? Reply with just the number (1, 2, or 3) and a one-sentence reason.\n\n"
selection_prompt += "\n".join(f"{i+1}. {t}" for i, t in enumerate(candidates_text))

r_pick = client.models.generate_content(
    model=MODEL, contents=selection_prompt,
    config=cfg(temperature=0.0)
)
print(f"\nModel's pick: {get_text(r_pick)}")

---
## 14. Multimodal Input (Vision)

Modern LLMs are **multimodal** — they can process images alongside text.

Use cases: image description, OCR from screenshots, chart analysis, photo Q&A, comparing two images.

The image is tokenised (broken into patches) and fed into the same transformer as the text.

In [ ]:
# Pass an image to the model — download it first then send as bytes
# (Gemma 4 requires bytes; some models support direct URL references)
image_url = "https://www.python.org/static/community_logos/python-logo-master-v3-TM.png"

with urllib.request.urlopen(image_url) as resp:
    img_bytes = resp.read()

r = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
        "What does this image show? Describe it in one sentence."
    ],
    config=cfg(temperature=0.3)
)
print("Image:", image_url)
print("Model:", get_text(r))

In [ ]:
# The img_bytes variable already holds the image from the cell above.
# This cell shows how to encode as base64 (useful for APIs that accept base64 strings,
# or for storing/caching images locally).
img_b64 = base64.b64encode(img_bytes).decode()
print(f"Base64 string length: {len(img_b64)} chars")
print(f"First 60 chars: {img_b64[:60]}...")

# Use the base64 bytes directly with from_bytes
r = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=base64.b64decode(img_b64), mime_type="image/png"),
        "What colours appear in this logo?"
    ],
    config=cfg(temperature=0.0)
)
print("\nModel:", get_text(r))

In [ ]:
# ✏️  To use a LOCAL image file, uncomment and set your path:
# with open("my_screenshot.png", "rb") as f:
#     img_bytes = f.read()
# r = client.models.generate_content(
#     model=MODEL,
#     contents=[
#         types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
#         "Describe what you see."
#     ],
#     config=cfg()
# )
# print(get_text(r))
print("(Uncomment and set a local image path to try this)")

---
## 15. Prompt Templates & Parameterization

Hard-coding prompts works for experiments. Real applications need **reusable, parameterized prompts** where only the data changes.

Benefits: reusability, consistency, testability, maintainability.

In [ ]:
# Level 1: Simple f-string template
def summarize(text: str, max_words: int = 50) -> str:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Summarize the following text in at most {max_words} words:\n\n{text}",
        config=cfg(temperature=0.3)
    )
    return get_text(r)

article = """
Python's popularity has surged over the past decade, becoming the go-to language for 
data science, AI, web development, and automation. Its readable syntax lowers the 
barrier for beginners, while its rich library ecosystem satisfies expert needs. 
Major tech companies like Google, Netflix, and NASA use it extensively.
"""

print("50-word summary:", summarize(article, max_words=50))
print()
print("20-word summary:", summarize(article, max_words=20))

In [ ]:
# Level 2: Template class — bundles system instruction + prompt + config
class PromptTemplate:
    def __init__(self, template: str, system_instruction: str = None,
                 temperature: float = 0.5, max_output_tokens: int = DEFAULT_MAX):
        self.template = template
        self.system_instruction = system_instruction
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def run(self, **kwargs) -> str:
        prompt = self.template.format(**kwargs)
        r = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=cfg(system_instruction=self.system_instruction,
                       temperature=self.temperature,
                       max_output_tokens=self.max_output_tokens)
        )
        return get_text(r)


# Define reusable templates
translate_template = PromptTemplate(
    template="Translate the following text to {language}:\n\n{text}",
    temperature=0.2
)

code_review_template = PromptTemplate(
    template="Review this {language} code and list any issues:\n\n```{language}\n{code}\n```",
    system_instruction="You are a senior software engineer. Be concise and specific.",
    temperature=0.2
)

# Use the templates
print("=== Translation ===")
print(translate_template.run(language="French", text="Hello, how are you today?"))

print("\n=== Code Review ===")
print(code_review_template.run(language="python", code="""
def divide(a, b):
    return a / b

result = divide(10, 0)
print(result)
"""))

In [ ]:
# Level 3: Chaining templates — output of one feeds into the next
extract_template = PromptTemplate(
    template="Extract the 3 most important facts from this text, one per line:\n\n{text}",
    temperature=0.1
)

tweet_template = PromptTemplate(
    template="Turn these facts into a single engaging tweet (max 280 chars, no hashtags):\n\n{facts}",
    temperature=0.8
)

news_article = """
Researchers at MIT have developed a new battery technology that charges 10x faster 
than current lithium-ion batteries. The battery uses a novel solid-state electrolyte 
that enables faster ion movement. It has been tested for 1,000 charge cycles with 
less than 5% capacity degradation. Commercial availability is expected by 2026.
"""

facts = extract_template.run(text=news_article)
print("Step 1 — Extracted facts:")
print(facts)

tweet = tweet_template.run(facts=facts)
print(f"\nStep 2 — Tweet ({len(tweet)} chars):")
print(tweet)

---
## 16. Key Takeaways

### Complete concepts cheat-sheet

| Concept | One-liner |
|---|---|
| LLM | Next-token predictor trained on massive text data |
| Token | ~3–4 chars; unit of cost, speed, and context |
| Thinking model | Reasons internally before answering (Gemma 4, o1, etc.) — needs higher `max_output_tokens` |
| Temperature | Randomness: 0 = deterministic, 2 = chaotic |
| top_k | Hard limit on candidate token count |
| top_p | Keep tokens until cumulative probability ≥ p |
| max_output_tokens | Hard ceiling on response length (includes thinking tokens) |
| stop_sequences | Custom strings that halt generation |
| System instruction | Persona/rules set before conversation starts |
| Context window | Total tokens the model can see at once |
| Stateless | Each call is independent; send history every time |
| Zero-shot | Ask without examples |
| Few-shot | Include 2–5 examples to teach format |
| Chain-of-thought | "Think step by step" — improves reasoning |
| Structured output | Force valid JSON matching a schema |
| Streaming | Receive tokens as generated, not all at once |
| Embedding | Dense vector representing the meaning of text |
| Cosine similarity | How semantically close two embeddings are (0–1) |
| Hallucination | Model confidently generating false information |
| Grounding | Provide facts in prompt; instruct model to use only those |
| Chunking | Split long docs to fit within context window |
| candidate_count | Get N independent responses in one call |
| Multimodal | Model processes both text and images |
| Prompt template | Reusable prompt with runtime variables |

### Suggested next experiments

1. **Parameter sweep** — run the same creative prompt at 5 temperatures, plot output length vs temperature
2. **Interactive chatbot** — wrap chat in `while True: user_input = input("You: ")`
3. **Semantic search engine** — embed a set of documents, find top-3 matches for any query
4. **Hallucination detector** — two-step: generate an answer, then verify it
5. **Document Q&A** — chunk a PDF, embed the chunks, retrieve relevant ones, answer questions
6. **Streaming UI** — build a simple Gradio app that streams responses to a web browser

### Resources
- [Gemini API docs](https://ai.google.dev/gemini-api/docs)
- [Gemma model card](https://ai.google.dev/gemma/docs)
- [Google AI Studio](https://aistudio.google.com) — interactive playground
- [Prompt engineering guide](https://ai.google.dev/gemini-api/docs/prompting-strategies)
- [Embeddings guide](https://ai.google.dev/gemini-api/docs/embeddings)